# 2-Step Symphony Pipeline Tutorial

In this tutorial, we will use example data to run the 2-Step Symphony Pipeline.

To run the example, use this command under the directory `Automatic-Cell-Typing/documents/symphony_tutorial/`.

```bash
Rscript ./xenium_symphony_pipeline.R ./symphony_tut.config.R
```

The first file is the 2-Step Symphony Pipeline, which is same as the one under the folder `Automatic-Cell-Typing/R/`. The second file is the configuration of 2-Step Symphony Pipeline, and requires users to modify it. The key point is that correctly set up the configuration. This tutorial will show how to do it.

## Look into the data for references

In folder `example_data/`, we have 2 Giotto objects. One is for references. Another is for queries.

Have a look at the metadata of the data for references.

In [4]:
giotto_obj_refer <- readRDS('example_data/example_refer_giotto.rds')
str(giotto_obj_refer@cell_metadata)		# metadata of cells in Giotto

Classes ‘data.table’ and 'data.frame':	3614 obs. of  8 variables:
 $ cell_ID        : chr  "Pan_T7980361_GTCAAGTTCACTGGGC" "Pan_T7991595_AAAGTAGCAAGCTGGA" "Pan_T7980361_TGGGCGTAGTGTTAGA" "Pan_T7991595_CTAAGACGTGTTGGGA" ...
 $ cell_id        : chr  "Pan_T7980361_GTCAAGTTCACTGGGC" "Pan_T7991595_AAAGTAGCAAGCTGGA" "Pan_T7980361_TGGGCGTAGTGTTAGA" "Pan_T7991595_CTAAGACGTGTTGGGA" ...
 $ sample_id      : chr  "A36-BLD" "A35-BLD" "A36-BLD" "A35-BLD" ...
 $ main_cell_type : chr  "Monocyte" "Monocyte" "Monocyte" "Monocyte" ...
 $ cell_type      : chr  "CD14 monocyte" "CD14 monocyte" "CD14 monocyte" "CD14 monocyte" ...
 $ umap_1         : num  7.45 -9.27 7.25 -8.65 4.9 ...
 $ umap_2         : num  -9.084 0.836 -10.655 0.268 -12.455 ...
 $ cell_type_score: num  0.898 0.904 0.898 0.904 0.861 0.898 0.904 0.905 0.861 0.861 ...
 - attr(*, ".internal.selfref")=<externalptr> 


In the metadata, main celltypes are in the column 'main_cell_type', and sub-celltypes are in the column 'cell_type'. Therefore, we set 

```R
maintype_col_name = 'main_cell_type'
subtype_col_name = 'cell_type' 
```

The column 'sample_id' defines which sample the cell comes from. Specifying it in `vars_use` helps the algorithm to better integrate samples from different samples and datasets.

```R
vars_use = c('sample_id')
```

## Set output folders

Here we store all references in a folder, and store analysis results in another folder. The pipeline will creat the folder if the folder doesn't exist.

```R
save_main_ref_dir = '/home/yulicai/symphony/Automatic-Cell-Typing/documents/symphony_tutorial/references'   # these 4 should be absolute paths
save_main_uwot_dir = save_main_ref_dir
save_sub_ref_dir = '/home/yulicai/symphony/Automatic-Cell-Typing/documents/symphony_tutorial/references'
save_sub_uwot_dir = save_sub_ref_dir

k = 5   # the k of KNN in mapping.

output_dir <- './outputs'   # directory for saving analyzing results
```

## Read inputs for references

This part have some code to read inputs. Because our input is an Giotto object, uncomment the code for Giotto and modify the path. Comment other code. It will looks like:

```R
########## Read Inputs For References ##########

## if provide a giotto object directly
library('Giotto')
giotto_object = readRDS('./example_data/example_refer_giotto.rds')

giotto_object = normalizeGiotto(giotto_object, norm_methods = "standard", logbase=exp(1), scalefactor = 10000, scale_genes = FALSE, scale_cells = FALSE)
ref_exp = giotto_object@norm_expr
ref_metadata = giotto_object@cell_metadata


## if provide expression matrix and metadata seperately
#ref_exp_path = ''
#ref_metadata_path = ''
#
#ref_exp = readRDS(ref_exp_path)	
#ref_metadata = readRDS(ref_metadata_path)
## If the expression matric is raw counts, do normalization.
#seurat_obj = CreateSeuratObject(ref_exp)
#seurat_obj = NormalizeData(seurat_obj)     # log(CP10K + 1) normalization


## if provide a seurat object directly
#seurat_obj = readRDS('')
#
#seurat_obj = NormalizeData(seurat_obj)     # log(CP10K + 1) normalization
#ref_exp = seurat_obj$RNA@data    # gene x cell, should be log(CP10K + 1) normalized 
#ref_metadata = seurat_obj@meta.data
```

## Read inputs for queries

Because our input for queries is an Giotto object, uncomment the code for Giotto and modify the path. Comment other code. It will looks like:

```R
########## Read Inputs For Queries ##########

### If provide full paths of all cell_features_matrix.h5 file of the xenium data
#query_paths <- as.matrix(read.table("xxxxx"))[,1]
##query_paths <- c('h5_file1', 'h5_file2')
#for (path in query_paths){
#    h5f <- Read10X_h5(path)
#    seurat_obj <- CreateSeuratObject(h5f$`Gene Expression`) # Not sure if it is applied to other h5 files
#    seurat_obj <- NormalizeData(seurat_obj)
#    seurat_objs <- c(seurat_objs, seurat_obj)
#}


## If provide a giotto object
library('Giotto')
giotto_object <- readRDS('./example_data/example_refer_giotto.rds')
seurat_obj <- CreateSeuratObject(giotto_object@raw_exprs)
seurat_obj <- NormalizeData(seurat_obj)
seurat_objs <- c()
seurat_objs <- c(seurat_objs, seurat_obj)
```

## Run the pipeline

```bash
Rscript ./xenium_symphony_pipeline.R ./symphony_tut.config.R
```

In [14]:
giotto_object = readRDS('./example_data/example_refer_giotto.rds')

giotto_object = normalizeGiotto(giotto_object, norm_methods = "standard", logbase=exp(1), scalefactor = 10000, scale_genes = FALSE, scale_cells = FALSE)
ref_exp = giotto_object@norm_expr
ref_metadata = giotto_object@cell_metadata